# 02 — Exploratory Data Analysis

MANGO Climate Twin — preprocessed datasets (2013, India land grid).

Run `scripts/eda_analysis.py` to regenerate figures in `docs/eda_figures/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
FIG = Path("../docs/eda_figures")
FIG.mkdir(parents=True, exist_ok=True)

In [ ]:
rain = pd.read_csv("../data/processed/rainfall_final.csv", parse_dates=["date"])
maxt = pd.read_csv("../data/processed/maxtemp_final.csv", parse_dates=["date"])
mint = pd.read_csv("../data/processed/mintemp_final.csv", parse_dates=["date"])
merged = pd.read_csv("../data/processed/climate_merged.csv", parse_dates=["date"])

merged["diurnal_range"] = merged["max_temp"] - merged["min_temp"]

print(f"Rainfall: {len(rain):,} rows | Temp merged: {len(merged):,} rows")
print(f"Dates: {rain.date.min().date()} → {rain.date.max().date()}")
print(f"Rain grid (0.25°): {rain.groupby(['latitude','longitude']).ngroups:,} cells")
print(f"Temp grid (1°): {maxt.groupby(['latitude','longitude']).ngroups:,} cells")

## 1. Seasonality

In [ ]:
month_labels = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

monthly_rain = rain.groupby("month").apply(lambda x: x.groupby("date").rainfall.sum().mean())
axes[0].bar(month_labels, monthly_rain.values, color="steelblue")
axes[0].set_title("Mean daily national rainfall by month")
axes[0].set_ylabel("mm (summed over all cells)")

axes[1].plot(month_labels, maxt.groupby("month").max_temp.mean(), "r-o", label="Max")
axes[1].plot(month_labels, mint.groupby("month").min_temp.mean(), "b-o", label="Min")
axes[1].set_title("National mean temperature by month")
axes[1].set_ylabel("°C")
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Spatial patterns

In [ ]:
rain_mean = rain.groupby(["latitude", "longitude"]).rainfall.mean().reset_index()
pivot = rain_mean.pivot(index="latitude", columns="longitude", values="rainfall")

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(pivot, cmap="YlGnBu", ax=ax)
ax.set_title("Mean daily rainfall (mm) — 0.25° grid")
plt.tight_layout()
plt.show()

## 3. Correlations

In [ ]:
cols = ["rainfall", "max_temp", "min_temp", "diurnal_range"]
corr = merged[cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Pearson correlations (merged dataset)")
plt.tight_layout()
plt.show()

corr

## 4. Rainfall category breakdown

In [ ]:
rain.rain_category.value_counts().plot(kind="bar", color="steelblue", figsize=(8, 4))
plt.title("Rainfall category counts (cell-days)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 5. Daily time series (national aggregates)

In [ ]:
daily = merged.groupby("date").agg(
    rain=("rainfall", "sum"),
    tmax=("max_temp", "mean"),
    tmin=("min_temp", "mean"),
)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(daily.index, daily.rain, color="steelblue", alpha=0.7)
ax1.set_ylabel("National rain sum (mm)", color="steelblue")
ax2 = ax1.twinx()
ax2.plot(daily.index, daily.tmax, "r-", alpha=0.6, label="Max temp")
ax2.plot(daily.index, daily.tmin, "b-", alpha=0.6, label="Min temp")
ax2.set_ylabel("Temperature (°C)")
ax2.legend(loc="upper right")
ax1.set_title("2013 daily national aggregates")
plt.tight_layout()
plt.show()